<a href="https://colab.research.google.com/github/williamfaraday123/SC4001-Neural-Network/blob/main/Lim_Isaac_Part_B_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Question B1 (15 marks)

Real world datasets often have a mix of numeric and categorical features – this dataset is one example. To build models on such data, categorical features have to be encoded or embedded.

PyTorch Tabular is a library that makes it very convenient to build neural networks for tabular data. It is built on top of PyTorch Lightning, which abstracts away boilerplate model training code and makes it easy to integrate other tools, e.g. TensorBoard for experiment tracking.

For questions B1 and B2, the following features should be used:   
- **Numeric / Continuous** features: dist_to_nearest_stn, dist_to_dhoby, degree_centrality, eigenvector_centrality, remaining_lease_years, floor_area_sqm
- **Categorical** features: month, town, flat_model_type, storey_range



---



In [ ]:
!pip install "pytorch_tabular[extra]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.8/165.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.8/159.8 kB 5.4 MB/s eta 0:00:00
  Attempting uninstall: wandb
    Found existing installation: wandb 0.25.0
    Uninstalling wandb-0.25.0:
      Successfully uninstalled wandb-0.25.0


In [ ]:
SEED = 42

import os

import random
random.seed(SEED)

import numpy as np
np.random.seed(SEED)

import pandas as pd

from pytorch_tabular import TabularModel
from pytorch_tabular.models import CategoryEmbeddingModelConfig
from pytorch_tabular.config import (
    DataConfig,
    OptimizerConfig,
    TrainerConfig,
)

from sklearn.metrics import r2_score

import torch
import omegaconf
import collections
import typing

# The "Master List" for PyTorch 2.6+ security limits
torch.serialization.add_safe_globals([
    # Python Primitives and Collections
    dict, list, set, str, int, float, bool,
    collections.defaultdict,
    collections.OrderedDict,

    # Typing modules used in metadata
    typing.Any, typing.Union, typing.Optional, typing.List, typing.Dict,

    # OmegaConf (The config library)
    omegaconf.dictconfig.DictConfig,
    omegaconf.listconfig.ListConfig,
    omegaconf.nodes.AnyNode,
    omegaconf.base.ContainerMetadata,
    omegaconf.base.Metadata,

    # PyTorch Tabular specific configs
    CategoryEmbeddingModelConfig,
    DataConfig,
    OptimizerConfig,
    TrainerConfig,

    # Numpy types often found in state dicts
    np.dtype,
    # Depending on your numpy version, this might be needed:
    # np._core.multiarray.scalar
])


1.Divide the dataset (‘hdb_price_prediction.csv’) into train, validation and test sets by using entries from year 2019 and before as training data, year 2020 as validation data and year 2021 as test data.
**Do not** use data from year 2022 and year 2023.



In [ ]:
df = pd.read_csv('hdb_price_prediction.csv')

# YOUR CODE HERE

# Force 'year' to be numeric just in case it's stored as a string
df['year'] = pd.to_numeric(df['year'], errors='coerce')

# Check if the years actually exist in your data
print(f"Years found in dataset: {df['year'].unique()}")

# Splitting the data
train = df[df['year'] <= 2019].copy()
val = df[df['year'] == 2020].copy()
test = df[df['year'] == 2021].copy()


# Critical Check: Ensure these are NOT zero
print(f"Train size: {len(train)}")
print(f"Val size: {len(val)}")
print(f"Test size: {len(test)}")

if len(val) == 0:
    raise ValueError("Validation set is empty! Check if year 2020 exists in your CSV.")

# Define features
num_cols = [
    "dist_to_nearest_stn", "dist_to_dhoby", "degree_centrality",
    "eigenvector_centrality", "remaining_lease_years", "floor_area_sqm"
]
cat_cols = ["month", "town", "flat_model_type", "storey_range"]
target_col = "resale_price"

Years found in dataset: [2017 2018 2019 2020 2021 2022 2023]
Train size: 64057
Val size: 23313
Test size: 29057


2.Refer to the documentation of **PyTorch Tabular** and perform the following tasks: https://pytorch-tabular.readthedocs.io/en/latest/#usage
- Use **[DataConfig](https://pytorch-tabular.readthedocs.io/en/latest/data/)** to define the target variable, as well as the names of the continuous and categorical variables.
- Use **[TrainerConfig](https://pytorch-tabular.readthedocs.io/en/latest/training/)** to automatically tune the learning rate. Set batch_size to be 1024 and set max_epoch as 50.
- Use **[CategoryEmbeddingModelConfig](https://pytorch-tabular.readthedocs.io/en/latest/models/#category-embedding-model)** to create a feedforward neural network with 1 hidden layer containing 50 neurons.
- Use **[OptimizerConfig](https://pytorch-tabular.readthedocs.io/en/latest/optimizer/)** to choose Adam optimiser. There is no need to set the learning rate (since it will be tuned automatically) nor scheduler.
- Use **[TabularModel](https://pytorch-tabular.readthedocs.io/en/latest/tabular_model/)** to initialise the model and put all the configs together.

In [ ]:
# YOUR CODE HERE
data_config = DataConfig(
    target=[target_col],
    continuous_cols=num_cols,
    categorical_cols=cat_cols,
)

trainer_config = TrainerConfig(
    auto_lr_find=True,  # Automatically tune the learning rate
    batch_size=1024,
    max_epochs=50,
)

# FFNN with 1 hidden layer of 50 neurons
model_config = CategoryEmbeddingModelConfig(
    task="regression",
    layers="50",
    activation="ReLU",
)

optimizer_config = OptimizerConfig() # Default is Adam

tabular_model = TabularModel(
    data_config=data_config,
    model_config=model_config,
    optimizer_config=optimizer_config,
    trainer_config=trainer_config,
)


# Fit and Evaluate
# Override the default behavior for this session
torch.serialization.default_weights_only = False

# Evaluate the model

# 2. FIT FIRST (This builds the datamodule and trains the weights)
# Use the context manager here to allow the LR Finder to reload the checkpoint
with torch.serialization.safe_globals([dict, list, str, int, float]):
    tabular_model.fit(train=train, validation=val)

# 3. EVALUATE SECOND (Now that datamodule exists)
result = tabular_model.evaluate(test)
print(result)


2026-03-11 08:06:19,046 - {pytorch_tabular.tabular_model:145} - INFO - Experiment Tracking is turned off
INFO:lightning_fabric.utilities.seed:Seed set to 42
2026-03-11 08:06:19,082 - {pytorch_tabular.tabular_model:547} - INFO - Preparing the DataLoaders
2026-03-11 08:06:19,134 - {pytorch_tabular.tabular_datamodule:527} - INFO - Setting up the datamodule for regression task
2026-03-11 08:06:19,316 - {pytorch_tabular.tabular_model:598} - INFO - Preparing the Model: CategoryEmbeddingModel
2026-03-11 08:06:19,433 - {pytorch_tabular.tabular_model:341} - INFO - Preparing the Trainer
INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Expe

Finding best initial lr:   0%|          | 0/100 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=100` reached.
INFO:pytorch_lightning.utilities.rank_zero:Restoring states from the checkpoint path at /content/.lr_find_0a64208a-ba49-4769-94f9-a67988ee0be8.ckpt
INFO:pytorch_lightning.utilities.rank_zero:Restored all states from the checkpoint at /content/.lr_find_0a64208a-ba49-4769-94f9-a67988ee0be8.ckpt
INFO:pytorch_lightning.tuner.lr_finder:Learning rate set to 2.7542287033381663e-05
2026-03-11 08:06:34,141 - {pytorch_tabular.tabular_model:668} - INFO - Suggested LR: 2.7542287033381663e-05. For plot and detailed analysis, use `find_learning_rate` method.
2026-03-11 08:06:34,143 - {pytorch_tabular.tabular_model:677} - INFO - Training Started


┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type                      ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ _backbone        │ CategoryEmbeddingBackbone │  2.9 K │ train │     0 │
│ 1 │ _embedding_layer │ Embedding1dLayer          │  1.5 K │ train │     0 │
│ 2 │ head             │ LinearHead                │     51 │ train │     0 │
│ 3 │ loss             │ MSELoss                   │      0 │ train │     0 │
└───┴──────────────────┴───────────────────────────┴────────┴───────┴───────┘

Trainable params: 4.5 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 4.5 K                                                                                                
Total estimated model params size (MB): 0                                                                          
Modules in train mode: 16                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

...

2026-03-11 08:10:07,139 - {pytorch_tabular.tabular_model:690} - INFO - Training the model completed
2026-03-11 08:10:07,142 - {pytorch_tabular.tabular_model:1531} - INFO - Loading the best model


Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_loss         │      287991529472.0       │
│  test_mean_squared_error  │      287991529472.0       │
└───────────────────────────┴───────────────────────────┘

[{'test_loss': 287991529472.0, 'test_mean_squared_error': 287991529472.0}]


3.Report the test MSE error and the test R2 value that you obtained.



In [ ]:
# 1. Generate predictions
predictions_df = tabular_model.predict(test)
print("Columns in prediction dataframe:", predictions_df.columns)


Columns in prediction dataframe: Index(['resale_price_prediction'], dtype='object')


In [ ]:
# YOUR CODE & RESULT HERE

# 1. Generate predictions
y_true = test['resale_price']
y_pred = predictions_df['resale_price_prediction']

# 2. Calculate R2
test_r2 = r2_score(y_true, y_pred)

# Extracting the values
metrics = result[0]
test_mse = metrics['test_mean_squared_error']

print(f"Test MSE: {test_mse:,.4f}")
print(f"Test R2 Value: {test_r2:.4f}")

Test MSE: 287,991,529,472.0000
Test R2 Value: -9.8873


4.Print out the corresponding rows in the dataframe for the top 20 test samples with the largest errors. Are there any patterns or common characteristics among these data points? Based on your observations, suggest and briefly explain a potential strategy to improve the model on these cases.



In [ ]:
# YOUR CODE & RESULT HERE
test_with_pred = test.copy()
test_with_pred['prediction'] = predictions_df['resale_price_prediction']
test_with_pred['error'] = abs(test_with_pred['resale_price'] - test_with_pred['prediction'])

top_20_errors = test_with_pred.sort_values(by='error', ascending=False).head(20)
pd.DataFrame(top_20_errors)

,month,year,town,full_address,nearest_stn,dist_to_nearest_stn,dist_to_dhoby,degree_centrality,eigenvector_centrality,flat_model_type,remaining_lease_years,floor_area_sqm,storey_range,resale_price,prediction,error
90608,12,2021,BISHAN,273B BISHAN STREET 24,Bishan,0.776182,6.297489,0.033613,0.015854,"5 ROOM, DBSS",88.833333,120.0,37 TO 39,1360000.0,2.891363,1.359997e+06
106199,12,2021,QUEENSTOWN,92 DAWSON ROAD,Queenstown,0.584731,3.882019,0.016807,0.008342,"5 ROOM, Premium Apartment Loft",93.333333,122.0,40 TO 42,1328000.0,4.643355,1.327995e+06
90483,9,2021,BISHAN,273A BISHAN STREET 24,Bishan,0.767244,6.327956,0.033613,0.015854,"5 ROOM, DBSS",89.000000,120.0,37 TO 39,1295000.0,4.889020,1.294995e+06
93931,12,2021,CENTRAL AREA,1B CANTONMENT ROAD,Outram Park,0.352779,2.413099,0.033613,0.121082,"5 ROOM, Type S2",88.083333,107.0,40 TO 42,1288000.0,5.689346,1.287994e+06
93930,12,2021,CENTRAL AREA,1D CANTONMENT ROAD,Outram Park,0.438348,2.506568,0.033613,0.121082,"5 ROOM, Type S2",88.166667,107.0,46 TO 48,1280000.0,6.224120,1.279994e+06
90432,8,2021,BISHAN,275A BISHAN STREET 24,Bishan,0.827889,6.370404,0.033613,0.015854,"5 ROOM, DBSS",88.916667,120.0,25 TO 27,1280000.0,6.917778,1.279993e+06
100836,6,2021,KALLANG/WHAMPOA,39 JALAN BAHAGIA,Boon Keng,0.998313,3.304953,0.016807,0.053004,"3 ROOM, Terrace",50.083333,210.0,01 TO 03,1268000.0,1.831575,1.267998e+06
101237,11,2021,KALLANG/WHAMPOA,8 BOON KENG ROAD,Bendemeer,0.352251,2.587444,0.016807,0.004414,"5 ROOM, DBSS",88.250000,119.0,40 TO 42,1268000.0,2.434387,1.267998e+06
93904,11,2021,CENTRAL AREA,1C CANTONMENT ROAD,Outram Park,0.401367,2.445314,0.033613,0.121082,"5 ROOM, Type S2",88.333333,106.0,40 TO 42,1261000.0,6.128881,1.260994e+06
90523,10,2021,BISHAN,273B BISHAN STREET 24,Bishan,0.776182,6.297489,0.033613,0.015854,"5 ROOM, DBSS",88.916667,120.0,22 TO 24,1260000.0,7.799953,1.259992e+06
